# Final Training — Qwen3.6-35B-A3B Full Data SFT

FINAL_TRAIN_ON_FULL_DATA=True: validation 없이 전체 데이터 학습

**업로드 필요 파일:**
- `my_code_0514from0508/` 폴더 전체
- `data/train.csv`, `data/test.csv`, `data/somenna_submission.csv`

## 1. Install Dependencies
**실행 후 Runtime > Restart Session 필수**

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --upgrade "transformers==5.5.0" "trl==0.24.0" "datasets<4.4.0"
!pip install cut_cross_entropy hf_transfer msgspec tyro peft accelerate bitsandbytes xformers
!pip install flash-attn --no-build-isolation
!pip install pandas tqdm scikit-learn sentence-transformers

## 2. 파일 업로드
파일 패널(왼쪽)에서 직접 업로드하거나 아래 셀로 업로드

In [ ]:
# 파일 업로드 UI (필요시 사용)
# from google.colab import files
# uploaded = files.upload()

# 업로드 후 디렉토리 구조 확인
import os
os.chdir('/content')
!ls -la

## 3. GPU 확인

In [ ]:
import torch
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} | VRAM: {vram:.1f} GB')

## 4. 플래그 패치: FINAL_TRAIN_ON_FULL_DATA=True

In [ ]:
import re

# 업로드한 폴더명에 맞게 경로 수정
SRC_DIR = '/content/my_code_0514from0508'  # 실제 업로드 폴더명
train_py = f'{SRC_DIR}/train.py'

with open(train_py, 'r', encoding='utf-8') as f:
    src = f.read()

src = re.sub(
    r'FINAL_TRAIN_ON_FULL_DATA\s*=\s*False',
    'FINAL_TRAIN_ON_FULL_DATA  = True',
    src
)

with open(train_py, 'w', encoding='utf-8') as f:
    f.write(src)

!grep 'FINAL_TRAIN_ON_FULL_DATA' {train_py}

## 5. 학습 실행

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, f'{SRC_DIR}/train.py'],
    cwd='/content',
)
print('Return code:', result.returncode)

## 6. 추론 실행

In [ ]:
# lora_model/ 이 /content/에 있어야 함
!ls /content/lora_model/

result = subprocess.run(
    [sys.executable, f'{SRC_DIR}/inference.py'],
    cwd='/content',
)
print('Return code:', result.returncode)

## 7. 제출 파일 다운로드

In [ ]:
from google.colab import files

# 경로 확인
!find /content -name 'submission.csv' 2>/dev/null

sub_path = '/content/submission.csv'
if os.path.exists(sub_path):
    files.download(sub_path)
else:
    alt = '/content/artifacts/submission.csv'
    if os.path.exists(alt):
        files.download(alt)
    else:
        print('submission.csv 위치를 위 find 결과로 확인하세요')